In [ ]:
import pandas as pd

# Read data from Excel file
excel_file = 'icd10cm.xlsx'  # Update with your file path
df = pd.read_excel(excel_file)


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# ==========================
# 1) Conexão com o MySQL
# ==========================
engine = create_engine("mysql+pymysql://root:@localhost/codificacao")

# ==========================
# 2) Renomear colunas do DataFrame
# ==========================
df_transformado = df.rename(columns={
    'Código': 'codigo',
    'Descrição PT_(Longa)': 'descricao_longa',
    'Descrição PT_(Curta)': 'descricao_curta',
    'Capitulo ICD-10-CM_ Código': 'capitulo_codigo',
    'Capitulo ICD-10-CM_desc': 'capitulo_descricao',
    'Capitulo ICD-10-CM_desc_PT': 'capitulo_descricao_pt',
    'Secção ICD-10-CM_Código': 'secao_codigo',
    'Secção ICD-10-CM_Desc': 'secao_descricao',
    'Secção ICD-10-CM_Desc_PT': 'secao_descricao_pt',
    'Válido': 'valido',
    'Ano inicio ': 'ano_inicio',
    'Ano fim': 'ano_fim',
    'Versão': 'versao',
    'Codigo versão anterior': 'codigo_versao_anterior',
    'Tipo Alteração': 'tipo_alteracao'
})

# ==========================
# 3) Selecionar colunas desejadas
# ==========================
colunas_desejadas = [
    'codigo','descricao_longa','descricao_curta','valido',
    'versao','codigo_versao_anterior','tipo_alteracao'
]

df_final = df_transformado[colunas_desejadas]

print("Colunas finais:", df_final.columns.tolist())

# ==========================
# 4) Criar tabela temporária no MySQL
# ==========================
df_final.to_sql(
    name='icd10cms_temp',
    con=engine,
    if_exists='replace',
    index=False
)

print("Tabela temporária criada.")


# ==========================
# 5) Criar tabela definitiva com ID AUTO_INCREMENT
#    (desativando FKs temporariamente)
# ==========================
with engine.connect() as conn:
    # Desativar foreign key checks
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))

    # Garantir remoção segura das tabelas
    conn.execute(text("DROP TABLE IF EXISTS icd10cms;"))
    conn.execute(text("DROP TABLE IF EXISTS icd10cms_new;"))

    # Reativar FKs
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

    # Criar a nova tabela com PK
    conn.execute(text("""
        CREATE TABLE icd10cms_new (
            id INT AUTO_INCREMENT PRIMARY KEY,
            codigo TEXT NULL,
            descricao_longa TEXT NULL,
            descricao_curta TEXT NULL,
            valido TEXT NULL,
            versao TEXT NULL,
            codigo_versao_anterior TEXT NULL,
            tipo_alteracao TEXT NULL
        );
    """))

print("Tabela icd10cms_new criada com chave primária.")


# ==========================
# 6) Inserir os dados
# ==========================
with engine.connect() as conn:
    conn.execute(text("""
        INSERT INTO icd10cms_new
        (codigo, descricao_longa, descricao_curta, valido, versao, codigo_versao_anterior, tipo_alteracao)
        SELECT codigo, descricao_longa, descricao_curta, valido, versao, codigo_versao_anterior, tipo_alteracao
        FROM icd10cms_temp;
    """))

print("Dados inseridos na tabela nova.")


# ==========================
# 7) Finalizar: renomear tabela
# ==========================
with engine.connect() as conn:
    conn.execute(text("DROP TABLE icd10cms_temp;"))
    conn.execute(text("ALTER TABLE icd10cms_new RENAME TO icd10cms;"))

print("Tabela final criada: icd10cms.")
print("Processo concluído com sucesso!")




In [9]:
import pandas as pd
from sqlalchemy import create_engine, text

# ==========================
# 1) Conexão com o MySQL usando SQLAlchemy
# ==========================
engine = create_engine("mysql+pymysql://root:@localhost/codificacao")

# ==========================
# 2) Renomear colunas do DataFrame
# ==========================
df_transformado = df.rename(columns={
    'Código': 'codigo',
    'Descrição PT_(Longa)': 'descricao_longa',
    'Descrição PT_(Curta)': 'descricao_curta',
    'Capitulo ICD-10-CM_ Código': 'capitulo_codigo',
    'Capitulo ICD-10-CM_desc': 'capitulo_descricao',
    'Capitulo ICD-10-CM_desc_PT': 'capitulo_descricao_pt',
    'Secção ICD-10-CM_Código': 'secao_codigo',
    'Secção ICD-10-CM_Desc': 'secao_descricao',
    'Secção ICD-10-CM_Desc_PT': 'secao_descricao_pt',
    'Válido': 'valido',
    'Ano inicio ': 'ano_inicio',
    'Ano fim': 'ano_fim',
    'Versão': 'versao',
    'Codigo versão anterior': 'codigo_versao_anterior',
    'Tipo Alteração': 'tipo_alteracao'
})

# ==========================
# 3) Selecionar colunas desejadas
# ==========================
colunas_desejadas = [
    'codigo','descricao_longa','descricao_curta','valido',
    'versao','codigo_versao_anterior','tipo_alteracao'
]

df_final = df_transformado[colunas_desejadas].copy()

# Converter NaN para None para compatibilidade com MySQL
df_final = df_final.where(pd.notna(df_final), None)

print("Colunas finais:", df_final.columns.tolist())
print("Linhas:", len(df_final))

# ==========================
# 4) Desativar constraints ANTES de criar tabela
# ==========================
with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
    conn.commit()

print("Foreign key checks desativadas.")

# ==========================
# 5) Remover tabela antiga se existir
# ==========================
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS icd10cms"))
    conn.commit()

print("Tabela antiga removida.")

# ==========================
# 6) Criar tabela com estrutura correta (com ID como PK)
# ==========================
try:
    with engine.connect() as conn:
        # Criar tabela vazia com estrutura correta
        conn.execute(text("""
            CREATE TABLE icd10cms (
                id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT PRIMARY KEY,
                codigo TEXT NULL,
                descricao_longa TEXT NULL,
                descricao_curta TEXT NULL,
                valido BIGINT NULL,
                versao TEXT NULL,
                codigo_versao_anterior TEXT NULL,
                tipo_alteracao TEXT NULL,
                created_at TIMESTAMP NULL,
                updated_at TIMESTAMP NULL
            )
        """))
        conn.commit()
    print("✅ Tabela icd10cms criada com estrutura correta.")
    
    # Agora inserir os dados
    df_final.to_sql(
        name='icd10cms',
        con=engine,
        if_exists='append',  # Usar append ao invés de replace
        index=False,
        method='multi',
        chunksize=1000
    )
    print("✅ Dados inseridos na tabela icd10cms com sucesso!")
    
except Exception as e:
    print(f"❌ Erro: {e}")
    print(f"Tipo de erro: {type(e).__name__}")
finally:
    # ==========================
    # 7) Reativar constraints após inserção
    # ==========================
    with engine.connect() as conn:
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))
        conn.commit()
    print("Foreign key checks reativadas.")

print("✅ Processo concluído com sucesso!")


Colunas finais: ['codigo', 'descricao_longa', 'descricao_curta', 'valido', 'versao', 'codigo_versao_anterior', 'tipo_alteracao']
Linhas: 99577
Foreign key checks desativadas.
Tabela antiga removida.
✅ Tabela icd10cms criada com estrutura correta.
✅ Dados inseridos na tabela icd10cms com sucesso!
Foreign key checks reativadas.
✅ Processo concluído com sucesso!
✅ Dados inseridos na tabela icd10cms com sucesso!
Foreign key checks reativadas.
✅ Processo concluído com sucesso!
